In [2]:
!pip install openai faiss-cpu langgraph langchain-core PyMuPDF tiktoken -q

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 61.6 MB/s eta 0:00:00


In [9]:
!mkdir -p src
!wget -q https://raw.githubusercontent.com/joshab01/AutoRFP-AI/main/src/__init__.py -O src/__init__.py
!wget -q https://raw.githubusercontent.com/joshab01/AutoRFP-AI/main/src/agents.py -O src/agents.py

import sys
sys.path.insert(0, 'src')

print("agents.py loaded")

agents.py loaded


Notebook 03 — Agent Pipeline (THE CORE)
AutoRFP-AI: Multi-Agent RFP Response System with LangGraph

WHAT THIS DOES:
- Wires Parser -> Draft -> Compliance agents via LangGraph
- Runs the full end-to-end pipeline on a sample RFP
- Shows retry/dedup logs (the "bug fix" demo)
- Outputs a formatted response document

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

PREREQUISITES: Run Notebooks 01 and 02 first

### Setup — load everything from previous notebooks

In [10]:
import os
import sys
import json
import time
import numpy as np
import faiss
from pathlib import Path
from openai import OpenAI

# Add src/ to import path
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
sys.path.insert(0, "src")
# If running from notebooks/ directory in Colab, also try parent
sys.path.insert(0, "..")

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

client = OpenAI()

# Load knowledge base
with open(DATA_DIR / "past_responses.json") as f:
    knowledge_base = json.load(f)
print(f"Knowledge base: {len(knowledge_base)} Q&A pairs")

# Load FAISS index
index = faiss.read_index(str(DATA_DIR / "faiss_index.bin"))
print(f"FAISS index: {index.ntotal} vectors")

# Load sample RFP
with open(DATA_DIR / "sample_rfp_text.txt") as f:
    sample_rfp_text = f.read()
print(f"Sample RFP: {len(sample_rfp_text)} chars")

Knowledge base: 50 Q&A pairs
FAISS index: 50 vectors
Sample RFP: 1880 chars


### Rebuild retrieve() function

In [11]:
EMBEDDING_MODEL = "text-embedding-3-small"


def get_embeddings(texts):
    response = client.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    return [item.embedding for item in response.data]


def retrieve(query, top_k=3, exclude_hashes=None):
    """Retrieve top-k similar past answers with dedup support."""
    if exclude_hashes is None:
        exclude_hashes = set()
    query_vec = np.array(get_embeddings([query]), dtype="float32")
    faiss.normalize_L2(query_vec)
    search_k = min(top_k + len(exclude_hashes) + 5, index.ntotal)
    scores, indices = index.search(query_vec, search_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        qa = knowledge_base[idx]
        if qa["hash"] in exclude_hashes:
            continue
        results.append({
            "id": qa["id"], "question": qa["question"],
            "answer": qa["answer"], "category": qa["category"],
            "score": float(score), "hash": qa["hash"],
        })
        if len(results) >= top_k:
            break
    return results


print("Retriever ready")

Retriever ready


### Initialize agents

In [12]:
from agents import ParserAgent, DraftAgent, ComplianceAgent

parser_agent = ParserAgent()
draft_agent = DraftAgent(retrieve_fn=retrieve)
compliance_agent = ComplianceAgent()

print("All agents initialized")

All agents initialized


### Define LangGraph pipeline

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class RFPState(TypedDict):
    """State flowing through the pipeline."""
    rfp_text: str
    requirements: list[dict]
    drafts: list[dict]
    compliance_results: list[dict]
    final_output: list[dict]
    processing_log: list[str]


def parse_rfp(state: RFPState) -> dict:
    """Node 1: Parse RFP into structured requirements."""
    log = list(state.get("processing_log", []))
    log.append(">>> PARSER AGENT: Extracting requirements...")

    requirements = parser_agent.parse(state["rfp_text"])
    log.append(f"    Extracted {len(requirements)} requirements")
    return {"requirements": requirements, "processing_log": log}


def draft_responses(state: RFPState) -> dict:
    """Node 2: Draft answers for each requirement."""
    log = list(state.get("processing_log", []))
    drafts = []
    total = len(state["requirements"])
    log.append(f"\n>>> DRAFT AGENT: Drafting {total} responses...")

    for i, req in enumerate(state["requirements"]):
        log.append(f"\n    [{i+1}/{total}] {req['id']}: {req['question'][:60]}...")

        draft = draft_agent.draft_single(
            question=req["question"],
            requirement_id=req["id"],
        )
        drafts.append(draft)

        log.append(f"    Confidence={draft['confidence']:.2f} "
                   f"Attempts={draft['attempts']} "
                   f"Chunks={draft['context_chunks_used']}"
                   + (" ** FLAGGED **" if draft["flagged_for_review"] else ""))

    flagged = sum(1 for d in drafts if d["flagged_for_review"])
    log.append(f"\n    Drafted {len(drafts)} responses ({flagged} flagged)")
    return {"drafts": drafts, "processing_log": log}


def check_compliance(state: RFPState) -> dict:
    """Node 3: Validate all drafts against compliance rules."""
    log = list(state.get("processing_log", []))
    results = []
    log.append(f"\n>>> COMPLIANCE AGENT: Checking {len(state['drafts'])} drafts...")

    for draft in state["drafts"]:
        result = compliance_agent.check(draft)
        results.append(result)
        if not result["passed"]:
            flags = ", ".join(f["rule_id"] for f in result["flags"])
            log.append(f"    [{draft['requirement_id']}] FLAGGED: {flags}")
        else:
            log.append(f"    [{draft['requirement_id']}] Passed")

    passed = sum(1 for r in results if r["passed"])
    log.append(f"\n    {passed}/{len(results)} passed compliance")
    return {"compliance_results": results, "processing_log": log}


def assemble_output(state: RFPState) -> dict:
    """Node 4: Merge drafts + compliance into final output."""
    log = list(state.get("processing_log", []))
    log.append("\n>>> ASSEMBLY: Merging final output...")

    final = []
    for draft, comp in zip(state["drafts"], state["compliance_results"]):
        final.append({
            "requirement_id": draft["requirement_id"],
            "question": draft["question"],
            "answer": draft["answer"],
            "confidence": draft["confidence"],
            "attempts": draft["attempts"],
            "flagged_for_review": draft["flagged_for_review"],
            "compliance_passed": comp["passed"],
            "compliance_flags": comp.get("flags", []),
        })

    log.append(f"    {len(final)} responses assembled")
    return {"final_output": final, "processing_log": log}


# Build the graph
workflow = StateGraph(RFPState)
workflow.add_node("parse", parse_rfp)
workflow.add_node("draft", draft_responses)
workflow.add_node("compliance", check_compliance)
workflow.add_node("assemble", assemble_output)

workflow.add_edge(START, "parse")
workflow.add_edge("parse", "draft")
workflow.add_edge("draft", "compliance")
workflow.add_edge("compliance", "assemble")
workflow.add_edge("assemble", END)

app = workflow.compile()

print("LangGraph pipeline compiled")
print("Flow: START -> Parse -> Draft -> Compliance -> Assemble -> END")

LangGraph pipeline compiled
Flow: START -> Parse -> Draft -> Compliance -> Assemble -> END


### Run the full pipeline

In [14]:
print("=" * 70)
print("RUNNING FULL AutoRFP PIPELINE")
print("=" * 70)

start_time = time.time()

result = app.invoke({
    "rfp_text": sample_rfp_text,
    "requirements": [],
    "drafts": [],
    "compliance_results": [],
    "final_output": [],
    "processing_log": [],
})

elapsed = time.time() - start_time

# Print processing log
print("\n-- Processing Log --")
for line in result["processing_log"]:
    print(line)

print(f"\nTotal time: {elapsed:.1f}s")

RUNNING FULL AutoRFP PIPELINE

-- Processing Log --
>>> PARSER AGENT: Extracting requirements...
    Extracted 12 requirements

>>> DRAFT AGENT: Drafting 12 responses...

    [1/12] REQ-001: What security protocols do you have in place to protect data...
    Confidence=0.90 Attempts=1 Chunks=3

    [2/12] REQ-002: How do you ensure the security of data in transit between ou...
    Confidence=0.90 Attempts=1 Chunks=3

    [3/12] REQ-003: Can you provide details on your incident response plan and h...
    Confidence=0.90 Attempts=1 Chunks=3

    [4/12] REQ-004: Describe the architecture of your cloud analytics platform. ...
    Confidence=0.90 Attempts=1 Chunks=3

    [5/12] REQ-005: What integrations do you offer with existing data sources an...
    Confidence=0.90 Attempts=1 Chunks=3

    [6/12] REQ-006: How does your platform handle real-time data processing and ...
    Confidence=0.90 Attempts=1 Chunks=3

    [7/12] REQ-007: What measures do you take to comply with global data privac

### Show retry/dedup log (THE BUG FIX DEMO)

In [15]:
print("\n" + "=" * 70)
print("RETRY & DEDUPLICATION LOG")
print("(The 'bug fix' from the LinkedIn post)")
print("=" * 70)

if draft_agent.retry_log:
    for entry in draft_agent.retry_log:
        req_id = entry["requirement_id"]
        if entry["action"] == "DRAFT":
            print(f"  [{req_id}] Attempt {entry['attempt']} -> "
                  f"confidence={entry['confidence']:.2f}, "
                  f"chunks={entry['chunks_retrieved']}, "
                  f"total_accumulated={entry['total_chunks']}")
        elif entry["action"] == "DEDUP_STOP":
            print(f"  [{req_id}] DEDUP STOP at attempt {entry['attempt']} -> "
                  f"overlap={entry['overlap_ratio']:.0%}")
else:
    print("  No retries occurred")

total_attempts = sum(e.get("attempt", 0) for e in draft_agent.retry_log if e["action"] == "DRAFT")
total_q = len(result["final_output"])
dedup_stops = sum(1 for e in draft_agent.retry_log if e["action"] == "DEDUP_STOP")
print(f"\n  {total_attempts} drafting attempts for {total_q} questions")
print(f"  {dedup_stops} dedup stops")
print(f"  Avg attempts/question: {total_attempts/max(total_q,1):.1f}")


RETRY & DEDUPLICATION LOG
(The 'bug fix' from the LinkedIn post)
  [REQ-001] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-002] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-003] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-004] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-005] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-006] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-007] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-008] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-009] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-010] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-011] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3
  [REQ-012] Attempt 1 -> confidence=0.90, chunks=3, total_accumulated=3

  12 drafting attempts for 12 questions
  0 dedup stops
  Avg attempt

### Render final output document

In [16]:
md = "# AutoRFP — Generated Response\n\n"
md += f"**Questions answered:** {len(result['final_output'])}  \n"
md += f"**Processing time:** {elapsed:.1f}s  \n\n---\n\n"

for entry in result["final_output"]:
    status = []
    if entry["flagged_for_review"]:
        status.append("NEEDS REVIEW")
    if not entry["compliance_passed"]:
        flags = ", ".join(f["rule_id"] for f in entry["compliance_flags"])
        status.append(f"Compliance: {flags}")
    status.append(f"Confidence: {entry['confidence']:.0%}")

    md += f"### {entry['requirement_id']}\n\n"
    md += f"**Q:** {entry['question']}\n\n"
    md += f"**A:** {entry['answer']}\n\n"
    md += f"*{' | '.join(status)} | Attempts: {entry['attempts']}*\n\n---\n\n"

# Display in Colab
try:
    from IPython.display import display, Markdown
    display(Markdown(md))
except ImportError:
    print(md)

# Save to file
output_path = DATA_DIR / "generated_rfp_response.md"
with open(output_path, "w") as f:
    f.write(md)
print(f"Saved to {output_path}")

# AutoRFP — Generated Response

**Questions answered:** 12  
**Processing time:** 76.4s  

---

### REQ-001

**Q:** What security protocols do you have in place to protect data stored in your cloud analytics platform?

**A:** A: Acme Cloud Platform implements a comprehensive suite of security protocols to protect data stored within our cloud analytics platform. We utilize AES-256 encryption for data at rest and TLS 1.2 for data in transit, ensuring that data remains secure during both storage and transmission. Our access control measures include role-based access control (RBAC) and multi-factor authentication (MFA), which enforce strict user permissions and significantly reduce the risk of unauthorized access. 

To further enhance security, we conduct regular security audits and vulnerability assessments, including third-party penetration testing every six months. Our commitment to security is also demonstrated through our adherence to industry standards and compliance frameworks, ensuring that our platform meets or exceeds regulatory requirements. We believe that robust security is foundational to our analytics offerings, providing peace of mind to our customers.

*Compliance: C-05, C-08 | Confidence: 90% | Attempts: 1*

---

### REQ-002

**Q:** How do you ensure the security of data in transit between our systems and your platform?

**A:** A: At Acme Cloud Platform, we prioritize the security of data in transit through a multi-layered approach. We utilize TLS 1.2 or higher for all data transmissions, ensuring that information is encrypted during transit to prevent unauthorized access. Our architecture also incorporates role-based access control (RBAC), which restricts access to sensitive data based on user roles, further enhancing security. Additionally, we conduct regular security audits and vulnerability assessments to identify and mitigate potential threats. Our commitment to security is reinforced by third-party penetration testing conducted bi-annually, ensuring our systems remain resilient against evolving threats.

*Confidence: 90% | Attempts: 1*

---

### REQ-003

**Q:** Can you provide details on your incident response plan and how you handle data breaches?

**A:** A: Acme Cloud Platform maintains a robust incident response plan designed to effectively manage and mitigate data breaches. Our dedicated incident response team is available 24/7 and follows a structured approach encompassing identification, containment, eradication, and recovery. In the event of a data breach, we commit to notifying affected clients within 72 hours of detection, providing detailed information about the breach, the data affected, and the remedial actions taken. We ensure ongoing communication through secure channels until the incident is fully resolved. Our average containment time for critical incidents is under 2 hours, minimizing disruption to our clients' operations. Additionally, we conduct regular incident response drills to ensure our team is prepared to handle various scenarios effectively.

*Confidence: 90% | Attempts: 1*

---

### REQ-004

**Q:** Describe the architecture of your cloud analytics platform. How does it support scalability and performance?

**A:** A: Acme Cloud Platform's cloud analytics architecture is built on a microservices framework using Kubernetes, which provides seamless scalability and high performance. Our platform supports independent scaling of services, allowing us to dynamically adjust resources based on demand. For the frontend, we utilize React, while our backend services are powered by Node.js and Python, ensuring efficient data processing and a responsive user experience.

To enhance performance, we implement PostgreSQL for relational data storage, combined with Redis for caching, which minimizes latency and maximizes availability. Our architecture supports up to 10,000 concurrent users per instance, with the ability to automatically provision additional instances as needed. Data ingestion is facilitated through batch uploads and real-time streaming via Apache Kafka, and our ETL pipeline leverages Apache Spark for large-scale data processing, enabling us to analyze millions of records per minute while maintaining data integrity.

*Compliance: C-01, C-02, C-04, C-05, C-06, C-07, C-08, C-09, C-10 | Confidence: 90% | Attempts: 1*

---

### REQ-005

**Q:** What integrations do you offer with existing data sources and third-party tools? Please provide examples.

**A:** A: Acme Cloud Platform provides extensive integration capabilities with existing data sources and third-party tools through our comprehensive suite of RESTful APIs. These APIs are designed to facilitate seamless data exchange and workflow automation, supporting both JSON and XML formats. Notable integrations include Salesforce for CRM data synchronization, Slack for team collaboration, and Google Workspace for document management and communication. Additionally, we offer Webhooks that allow real-time notifications of events and changes, ensuring that users are always up-to-date. Our developer portal contains detailed API documentation and example use cases to assist developers in implementing these integrations efficiently.

*Confidence: 90% | Attempts: 1*

---

### REQ-006

**Q:** How does your platform handle real-time data processing and analytics?

**A:** A: Acme Cloud Platform offers robust real-time data processing and analytics capabilities through a combination of advanced technologies. Data ingestion is facilitated by Apache Kafka, which supports real-time streaming and ensures that data flows seamlessly into our system. Once ingested, data is processed using Apache Spark within our ETL pipeline, allowing us to handle millions of records per minute with high efficiency.

Our platform is built on a microservices architecture utilizing Kubernetes, enabling independent scaling of services to meet varying workloads. For analytics, we leverage a combination of PostgreSQL for structured data storage and Redis for caching frequently accessed data, which minimizes latency and enhances performance. This architecture not only supports real-time analytics but also ensures that users can derive insights quickly and accurately, enabling data-driven decision-making.

*Confidence: 90% | Attempts: 1*

---

### REQ-007

**Q:** What measures do you take to comply with global data privacy regulations (e.g., GDPR, CCPA)?

**A:** A: Acme Cloud Platform prioritizes compliance with global data privacy regulations, including GDPR and CCPA. We adhere to key principles such as data minimization and purpose limitation, ensuring that customer data is collected and processed responsibly. Our platform includes built-in tools for managing Data Subject Requests (DSRs), enabling users to easily access, rectify, or delete their personal data.

To facilitate compliance audits, we provide clients with comprehensive documentation that demonstrates our adherence to data privacy requirements. A dedicated compliance liaison is available to assist with audit requests and documentation sharing. Additionally, we conduct regular internal audits and maintain certifications like ISO 27001 to assure clients of our commitment to data protection.

Our Data Processing Agreement (DPA) incorporates standard contractual clauses (SCCs) to ensure compliance with GDPR and CCPA. The DPA outlines the roles and obligations of both parties, includes data security measures, and establishes breach notification timelines within 72 hours. This transparency reinforces our dedication to upholding data privacy standards.

*Compliance: C-03, C-08 | Confidence: 90% | Attempts: 1*

---

### REQ-008

**Q:** How do you manage data ownership and access rights for our organization within your platform?

**A:** A: At Acme Cloud Platform, we prioritize data ownership and access rights through a comprehensive role-based access control (RBAC) system. This system ensures that access is granted based on the principle of least privilege, meaning users only have the permissions necessary for their roles. Organizations can define custom roles and permissions tailored to their specific needs, ensuring that sensitive data is accessible only to authorized personnel.

Access rights are regularly reviewed on a quarterly basis to adapt to any changes in personnel or organizational structure. Additionally, all access attempts are logged and monitored for suspicious activity, providing a clear audit trail for accountability. To enhance security, we also implement multi-factor authentication (MFA) for all user logins, reinforcing our commitment to safeguarding your data.

Our platform empowers organizations to manage their data ownership effectively while maintaining strict access controls to protect sensitive information.

*Confidence: 90% | Attempts: 1*

---

### REQ-009

**Q:** What is your typical timeline for implementing your cloud analytics platform from start to finish?

**A:** A: The typical timeline for implementing the Acme Cloud Analytics Platform ranges from 6 to 10 weeks, contingent upon several factors such as data volume, existing system complexity, and the level of customization required. The process begins with a kickoff meeting to align on objectives and establish a detailed project plan. Weeks 1-2 focus on environment setup and integration with existing data sources. Weeks 3-5 are dedicated to data migration and initial configuration of analytics workflows. During weeks 6-8, we conduct user acceptance testing (UAT) and training sessions to ensure your team is well-prepared. The final weeks are reserved for adjustments based on feedback and a phased rollout. Our dedicated project manager will maintain regular communication throughout to ensure a smooth implementation.

*Compliance: C-06 | Confidence: 90% | Attempts: 1*

---

### REQ-010

**Q:** What support and training do you provide during the implementation phase to ensure a smooth transition?

**A:** A: During the implementation phase, Acme Cloud Platform prioritizes a smooth transition through a structured support and training program. We begin with an initial onboarding session to align on project goals and timelines. Following this, clients receive a customized onboarding plan that includes access to self-paced training modules and live workshops tailored to specific user roles and workflows. 

To further enhance user adoption and minimize disruption, we implement a change management strategy that involves stakeholder engagement and regular communication updates. Our dedicated project manager and technical support specialist are available throughout the rollout, providing 24/7 support via email, chat, and phone. Additionally, a dedicated Slack channel is established for real-time troubleshooting and collaboration, ensuring that any issues are swiftly addressed.

Overall, our comprehensive approach ensures that users are well-equipped and supported during the transition to the Acme Cloud Platform.

*Confidence: 90% | Attempts: 1*

---

### REQ-011

**Q:** Can you outline your pricing model, including any tiered options based on usage or features?

**A:** A: Acme Cloud Platform employs a tiered pricing model designed to accommodate various business needs and usage levels. We offer three primary subscription tiers: Basic, Professional, and Enterprise. The Basic plan starts at $99 per month, suitable for startups needing essential features. The Professional plan, priced at $499 per month, includes advanced capabilities like data integration and workflow automation, catering to mid-sized organizations. The Enterprise plan is fully customizable, providing unlimited user licenses and dedicated support tailored to your specific requirements.

For businesses with fluctuating usage, we also provide a pay-as-you-go option, allowing you to pay based on actual consumption. This model is particularly beneficial for organizations with seasonal demands. Additionally, our usage dashboard assists in forecasting trends, enabling effective budget planning. We maintain transparency regarding potential additional fees, which may arise from exceeding data storage limits or API request thresholds, and we notify you proactively to avoid any surprises.

*Compliance: C-04 | Confidence: 90% | Attempts: 1*

---

### REQ-012

**Q:** Are there additional costs for support, maintenance, or upgrades? Please provide a detailed breakdown.

**A:** A: At Acme Cloud Platform, we prioritize transparency in our pricing structure. Our subscription plans encompass all standard support, maintenance, and upgrades, ensuring there are no hidden costs as you scale. This includes access to our knowledge base, community forums, and regular product updates at no additional charge. 

However, there may be additional costs associated with exceeding certain usage limits, such as data storage or API request thresholds, particularly if you are on a pay-as-you-go plan. We provide clear guidelines on these limits in our service agreement and will notify you proactively before any overage charges are incurred.

For customers needing advanced support, we offer three tiers: Standard, Premium, and Enterprise, each with varying response times and dedicated resources. Upgrades or downgrades in service levels can be made at any time, with changes taking effect in the next billing cycle. 

Overall, our commitment is to provide a clear and predictable pricing structure that aligns with your needs.

*Compliance: C-04 | Confidence: 90% | Attempts: 1*

---



Saved to data/generated_rfp_response.md


### Pipeline statistics

In [17]:
total = len(result["final_output"])
passed = sum(1 for e in result["final_output"] if e["compliance_passed"])
high_conf = sum(1 for e in result["final_output"] if e["confidence"] >= 0.7)
flagged = sum(1 for e in result["final_output"] if e["flagged_for_review"])
avg_conf = sum(e["confidence"] for e in result["final_output"]) / total

print("\n" + "=" * 70)
print("PIPELINE STATISTICS")
print("=" * 70)
print(f"  Requirements answered:   {total}")
print(f"  Compliance pass rate:    {passed}/{total} ({100*passed//total}%)")
print(f"  High-confidence (>=0.7): {high_conf}/{total} ({100*high_conf//total}%)")
print(f"  Flagged for review:      {flagged}/{total}")
print(f"  Average confidence:      {avg_conf:.2f}")
print(f"  Total time:              {elapsed:.1f}s ({elapsed/total:.1f}s per question)")
print(f"\nPipeline complete!")


PIPELINE STATISTICS
  Requirements answered:   12
  Compliance pass rate:    6/12 (50%)
  High-confidence (>=0.7): 12/12 (100%)
  Flagged for review:      0/12
  Average confidence:      0.90
  Total time:              76.4s (6.4s per question)

Pipeline complete!
